In [1]:
import pandas as pd

In [5]:
df_word_pairs = pd.read_excel("Experiments + Datasets.xlsx", sheet_name="Irish_Word_Pairs")

In [7]:
print(df_word_pairs.columns)
print(df_word_pairs.head())
print("Rows:", len(df_word_pairs))

Index(['Pair ID', 'Original Terms', 'Inclusive Terms', 'Original Terms GA',
       'Inclusive Terms GA'],
      dtype='object')
   Pair ID Original Terms Inclusive Terms    Original Terms GA  \
0        1      boyfriend         partner       buachaill cara   
1        2     boyfriends        partners    buachaillí cairde   
2        3         boyish        childish             óigeanta   
3        4        callboy      sex worker   buachaill fógartha   
4        5       callboys     sex workers  buachaillí fógartha   

  Inclusive Terms GA  
0        comhpháirtí  
1     comhpháirtithe  
2          páistiúil  
3     duine fógartha  
4    daoine fógartha  
Rows: 545


In [11]:
#Dataset Cleaning
print("Missing original terms:", df_word_pairs["Original Terms GA"].isna().sum())
print("Missing inclusive terms:", df_word_pairs["Inclusive Terms GA"].isna().sum())

duplicate_pairs = df_word_pairs.duplicated(subset=["Original Terms GA", "Inclusive Terms GA"]).sum()

print("Duplicate pairs:", duplicate_pairs)

Missing original terms: 0
Missing inclusive terms: 0
Duplicate pairs: 11


In [12]:
#Dropping Duplictes
df_word_pairs_clean = df_word_pairs.drop_duplicates(subset=["Original Terms GA", "Inclusive Terms GA"])

print("Original rows:", len(df_word_pairs))
print("Rows after removing duplicates:", len(df_word_pairs_clean))

Original rows: 545
Rows after removing duplicates: 534


In [15]:
triples = []

for _, row in df_word_pairs_clean.iterrows():
    gendered = row["Original Terms GA"]
    inclusive = row["Inclusive Terms GA"]

    triples.append((gendered, "HAS_INCLUSIVE_FORM", inclusive))

print("Number of unique triples:", len(triples))
print(triples[:10])

Number of unique triples: 534
[('buachaill cara', 'HAS_INCLUSIVE_FORM', 'comhpháirtí'), ('buachaillí cairde', 'HAS_INCLUSIVE_FORM', 'comhpháirtithe'), ('óigeanta', 'HAS_INCLUSIVE_FORM', 'páistiúil'), ('buachaill fógartha', 'HAS_INCLUSIVE_FORM', 'duine fógartha'), ('buachaillí fógartha', 'HAS_INCLUSIVE_FORM', 'daoine fógartha'), ('búachaill bó', 'HAS_INCLUSIVE_FORM', 'duine bó'), ('búachaillí bó', 'HAS_INCLUSIVE_FORM', 'daoine  bó'), ('buachaill iascaire', 'HAS_INCLUSIVE_FORM', 'iascaire óg'), ('buachaillí iascaire', 'HAS_INCLUSIVE_FORM', 'iascairí óga'), ('ardbhuachaill', 'HAS_INCLUSIVE_FORM', 'ceannaire mac léinn')]


In [17]:
#Checking does one exclusive term maps to multiple inclusive terms
multiple_mappings = (df_word_pairs_clean.groupby("Original Terms GA")["Inclusive Terms GA"].nunique().sort_values(ascending=False))
print(multiple_mappings.head(20))

Original Terms GA
fir an bhaile        2
buaiseach            2
mná sneachta         2
marcraí              2
marcach              2
ceann urra           2
bean-fhear           2
fear tionlacain      2
I gcoinne bean       2
mála fir             2
púca                 2
gíománach            1
grá idir chailíní    1
gnáthmhná            1
graifnóireachtaí     1
gíonmánaithe         1
graifnóireacht       1
iascaire ban         1
go cailínúil         1
iascairí ban         1
Name: Inclusive Terms GA, dtype: int64


In [18]:
#Networkx
import networkx as nx

In [21]:
G_pairs = nx.DiGraph()

for _, row in df_word_pairs_clean.iterrows():
    gendered = row["Original Terms GA"]
    inclusive = row["Inclusive Terms GA"]

    G_pairs.add_node(gendered,node_type="GenderedTerm")

    G_pairs.add_node(inclusive, node_type="InclusiveTerm")

    G_pairs.add_edge(gendered,inclusive,predicate="HAS_INCLUSIVE_FORM", layer="inclusive")

print("Nodes:", G_pairs.number_of_nodes())
print("Edges:", G_pairs.number_of_edges())

Nodes: 921
Edges: 534


In [23]:
#Pilot
def get_inclusive_forms(term):
    if term not in G_pairs:
        return []

    results = []

    for neighbour in G_pairs.successors(term):
        edge_data = G_pairs.get_edge_data(term, neighbour)

        if edge_data.get("predicate") == "HAS_INCLUSIVE_FORM":
            results.append(neighbour)

    return results

In [25]:
test_term = df_word_pairs_clean["Original Terms GA"].iloc[0]

print("Original term:", test_term)
print("Inclusive forms:", get_inclusive_forms(test_term))

Original term: buachaill cara
Inclusive forms: ['comhpháirtí']


In [27]:
#Pilot for terms with 2 inclusive objects:
multiple_terms = multiple_mappings[multiple_mappings > 1]

print(multiple_terms)

Original Terms GA
fir an bhaile      2
buaiseach          2
mná sneachta       2
marcraí            2
marcach            2
ceann urra         2
bean-fhear         2
fear tionlacain    2
I gcoinne bean     2
mála fir           2
púca               2
Name: Inclusive Terms GA, dtype: int64


In [28]:
test_multi = multiple_terms.index[0]

print("Original term:", test_multi)
print("Inclusive forms:", get_inclusive_forms(test_multi))

Original term: fir an bhaile
Inclusive forms: ['daoine baile', 'muintir an bhaile']
